# Popper — falsify the spec, then verify the proof

*Popper* is the system; **`falsify`** is the package that implements it. This
notebook runs the whole thing end to end:

1. the **numerical oracle** (math),
2. the **offline code-spec oracle**,
3. **M2** counterexample-guided repair, and
4. the **live Verina audit over the Axiom Lean Engine (AXLE)**.

A Lean checker answers *"is this proof valid?"*. Popper adds *"is this statement
faithful and worth proving?"* — an executable oracle that returns counterexamples.

In [1]:
import os
import sys

for p in (os.path.abspath("."), os.path.abspath("..")):
    if p not in sys.path:
        sys.path.insert(0, p)

import falsify
print("falsify", falsify.__version__)

falsify 0.3.0


## 1. Numerical oracle (math)

Faithful information-theory statements survive Monte-Carlo falsification;
unfaithful ones (a dropped hypothesis, a flipped direction) are refuted with the
concrete violating instance.

In [2]:
from falsify import NumericalOracle, information_theory_library, run_audit

rep = run_audit(information_theory_library(), NumericalOracle(),
                "Numerical oracle — information theory")
print(rep.render_terminal())


=== Numerical oracle — information theory ===
7 claims | ✓ FAITHFUL 3  ✗ FALSIFIED 3  ? INCONCLUSIVE 1
✓ [FAITHFUL    ] kl_nonneg: survived 2000 Monte-Carlo draws (2000 with hypothesis active)
✗ [FALSIFIED   ] kl_nonneg_DROPPED_normalization: counterexample found; the formalized statement is false as written  ⟵ q=[0.777, 1.534, 1.215] (Σq=3.525≠1) ⇒ Σ pᵢ·log(pᵢ/qᵢ) = -1.1543 < 0
✓ [FAITHFUL    ] data_processing: survived 2000 Monte-Carlo draws (2000 with hypothesis active)
✗ [FALSIFIED   ] data_processing_DROPPED_markov: counterexample found; the formalized statement is false as written  ⟵ I(X;Z) = 0.8318 > I(X;Y) = 0.0478  (Z leaks X directly)
✓ [FAITHFUL    ] entropy_concave: survived 2000 Monte-Carlo draws (2000 with hypothesis active)
✗ [FALSIFIED   ] entropy_concave_WRONG_direction: counterexample found; the formalized statement is false as written  ⟵ H(λp+(1-λ)q) = 1.0797 > λH(p)+(1-λ)H(q) = 1.0600  (entropy is concave, not convex)
? [INCONCLUSIVE] entropy_uniform_vacuous_guard:

## 2. Offline code-spec oracle

Soundness / completeness / vacuity on representative Verina-style tasks,
evaluated against an executable model (no Lean toolchain needed).

In [3]:
from falsify import CodeSpecOracle, MockAxleClient, verina_like_tasks

rep = run_audit(verina_like_tasks(), CodeSpecOracle(MockAxleClient()),
                "Code-spec oracle — offline fixtures")
print(rep.render_terminal())


=== Code-spec oracle — offline fixtures ===
4 claims | ✓ FAITHFUL 1  ✗ UNSOUND 1  ✗ INCOMPLETE 1  ✗ VACUOUS 1
✗ [VACUOUS     ] sort_by_length: spec constrains nothing — even the throwaway impl 'all_zeros' satisfies it  ⟵ impl 'all_zeros' passes; spec fails to pin down the answer
✗ [INCOMPLETE  ] max_lower_bound_only: spec is too weak — the wrong impl 'max_plus_one' satisfies it yet disagrees with the reference  ⟵ impl 'max_plus_one' passes the spec; differs from reference at input (3, 5)
✓ [FAITHFUL    ] abs_value: reference passes; no wrong implementation slips through; not vacuous
✗ [UNSOUND     ] abs_strictly_positive: spec is too strong — it rejects the correct reference implementation  ⟵ reference fails spec at input (0,) (→ output 0)



## 3. M2 — counterexample-guided repair

Each unfaithful spec is driven to FAITHFUL using the oracle's counterexample.

In [4]:
from falsify import default_repairer, repair_loop

oracle = CodeSpecOracle(MockAxleClient())
for task in verina_like_tasks():
    print(repair_loop(task, oracle, default_repairer()).render())

✓ sort_by_length: ✗VACUOUS  →  ✓FAITHFUL
      first counterexample: impl 'all_zeros' passes; spec fails to pin down the answer
✓ max_lower_bound_only: ✗INCOMPLETE  →  ✓FAITHFUL
      first counterexample: impl 'max_plus_one' passes the spec; differs from reference at input (3, 5)
✓ abs_value: ✓FAITHFUL
✓ abs_strictly_positive: ✗UNSOUND  →  ✓FAITHFUL
      first counterexample: reference fails spec at input (0,) (→ output 0)


## 4. Live Verina audit over AXLE

Real benchmark tasks, real Lean. Each task's `expected` / `unexpected` outputs
become `native_decide` witnesses checked through the Axiom Lean Engine:
**UNSOUND** if a correct output is rejected, **INCOMPLETE** if a wrong one is
accepted. Requires `pip install axiom-axle` and `AXLE_API_KEY`.

In [5]:
from falsify import run_live_audit

if os.environ.get("AXLE_API_KEY"):
    report = run_live_audit(limit=5, max_tests=1, max_unexpected=2, progress=False)
    print(report.render_terminal())
else:
    print("Set AXLE_API_KEY (and `pip install axiom-axle`) to run the live audit.")


=== Live Verina spec-faithfulness audit (AXLE) ===
5 claims | ✓ FAITHFUL 4  ? INCONCLUSIVE 1
✓ [FAITHFUL    ] verina_advanced_1: correct outputs accepted; all wrong outputs rejected (on test witnesses)
? [INCONCLUSIVE] verina_advanced_10: spec not decidable on some witnesses (no Decidable instance / timeout)
✓ [FAITHFUL    ] verina_advanced_11: correct outputs accepted; all wrong outputs rejected (on test witnesses)
✓ [FAITHFUL    ] verina_advanced_12: correct outputs accepted; all wrong outputs rejected (on test witnesses)
✓ [FAITHFUL    ] verina_advanced_13: correct outputs accepted; all wrong outputs rejected (on test witnesses)



## Honesty

Popper *falsifies*; it does not certify. **FAITHFUL** = no counterexample within
the search budget. **INCONCLUSIVE** = the spec isn't `Decidable` on some witness.
Lean/AXLE remains the ground truth for the *proof*.